In [1]:
import face_recognition
import cv2
import numpy as np
import os
import pandas as pd
from datetime import datetime

path = 'images'

images = []
classNames = []

myList = os.listdir(path)

for cl in myList:
    curImg = cv2.imread(f'{path}/{cl}')
    images.append(curImg)
    classNames.append(os.path.splitext(cl)[0])

def findEncodings(images):
    encodeList = []

    for img in images:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        encode = face_recognition.face_encodings(img)[0]
        encodeList.append(encode)

    return encodeList

def markAttendance(name):
    with open('attendance.csv', 'a') as f:
        now = datetime.now()
        dtString = now.strftime('%H:%M:%S')

        f.write(f'\n{name},{dtString}')

encodeListKnown = findEncodings(images)

print("Encoding Complete")

cap = cv2.VideoCapture(0)

while True:
    success, img = cap.read()

    imgS = cv2.resize(img, (0,0), None, 0.25, 0.25)
    imgS = cv2.cvtColor(imgS, cv2.COLOR_BGR2RGB)

    facesCurFrame = face_recognition.face_locations(imgS)
    encodesCurFrame = face_recognition.face_encodings(
        imgS, facesCurFrame)

    for encodeFace, faceLoc in zip(
            encodesCurFrame, facesCurFrame):

        matches = face_recognition.compare_faces(
            encodeListKnown, encodeFace)

        faceDis = face_recognition.face_distance(
            encodeListKnown, encodeFace)

        matchIndex = np.argmin(faceDis)

        if matches[matchIndex]:
            name = classNames[matchIndex].upper()

            y1,x2,y2,x1 = faceLoc

            y1,x2,y2,x1 = y1*4,x2*4,y2*4,x1*4

            cv2.rectangle(img,(x1,y1),(x2,y2),
                          (0,255,0),2)

            cv2.rectangle(img,(x1,y2-35),
                          (x2,y2),(0,255,0),
                          cv2.FILLED)

            cv2.putText(img,name,(x1+6,y2-6),
                        cv2.FONT_HERSHEY_COMPLEX,
                        1,(255,255,255),2)

            markAttendance(name)

    cv2.imshow('Webcam', img)

    if cv2.waitKey(1) == 13:
        break

cap.release()
cv2.destroyAllWindows()

ModuleNotFoundError: No module named 'face_recognition'

In [2]:
import face_recognition
print("Installed Successfully")

ModuleNotFoundError: No module named 'face_recognition'

In [3]:
import cv2

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    'haarcascade_frontalface_default.xml')

cap = cv2.VideoCapture(0)

while True:
    ret, img = cap.read()

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray, 1.1, 4)

    for (x, y, w, h) in faces:
        cv2.rectangle(img,
                      (x, y),
                      (x+w, y+h),
                      (255, 0, 0), 2)

    cv2.imshow('Face Detection', img)

    if cv2.waitKey(1) == 13:
        break

cap.release()
cv2.destroyAllWindows()